# Model housing dataset

In [ ]:
# IMport libaries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load dataset
df = pd.read_csv('https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv')

In [ ]:
df.head()
df.shape

## EDA

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
# check for null values
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
# PLot all numerical columns
num_cols = df.select_dtypes(include=np.number).columns

plt.figure(figsize=(16, 12))
for i, col in enumerate(num_cols, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[col], kde=True, bins=50)
    plt.title(col)
    plt.xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
""" 
- We need to fill nan values of total_bedrooms column with 
median value of the column.
- We need to convert ocean_proximity column to numerical values 
using one hot encoding.
- Replace totalrooms, totalbedrooms, population  
with rooms_per_household , bedrooms_per_rooms , bedrooms_per_household and
population per household columns.
"""


df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)
ohe_cols = df.columns[df.columns.str.startswith('ocean_proximity_')]
df[ohe_cols] = df[ohe_cols].astype(int)

df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

df.drop(['total_rooms', 'total_bedrooms'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
# Geographic distribution: house value vs each ocean_proximity category, in a grid
ocean_cols = [col for col in df.columns if col.startswith('ocean_proximity_')]
plots = ['median_house_value'] + ocean_cols

n_cols = 2
n_rows = -(-len(plots) // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14 * n_cols, 12 * n_rows))
axes = axes.flatten()

sns.scatterplot(data=df, x='longitude', y='latitude', hue='median_house_value',
                 palette='viridis', alpha=0.4, size='population', ax=axes[0])
axes[0].set_title('Median House Value', fontsize=20)
axes[0].tick_params(labelsize=13)

for i, col in enumerate(ocean_cols, start=1):
    sns.scatterplot(data=df, x='longitude', y='latitude', hue=col,
                     palette={0: 'lightgray', 1: 'red'}, alpha=0.5, s=40, ax=axes[i])
    axes[i].set_title(col, fontsize=20)
    axes[i].tick_params(labelsize=13)

for j in range(len(plots), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Check correlation between features and target variable
correlation_matrix = df.corr()
correlation_matrix['median_house_value'].sort_values(ascending=False)

# Visualise correlation with heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# we will drop longitude and latitude columns as they are not useful for prediction
# df.drop(['longitude', 'latitude'], axis=1, inplace=True)


In [ ]:
plt.figure(figsize=(10, 6))
subset = df[df['population_per_household'] < 10]  # drop extreme outliers for readability
sns.scatterplot(data=subset, x='population_per_household', y='median_house_value', alpha=0.3)
plt.title('Median House Value vs Population per Household', fontsize=16)
plt.xlabel('Population per Household', fontsize=14)
plt.ylabel('Median House Value', fontsize=14)
plt.show()

In [ ]:
df.head()

In [ ]:
# drop not required columsn for prediction
df.drop(['households', 'population', 'population_per_household'], axis=1, inplace=True)

In [ ]:
# we will consider fllowing features for prediction:
# - hosuing median age
# - housing median income
# - ocean porximity categories (one-hot encoded)
# - rooms per household
# - bedrooms per room

In [ ]:
# prepare test and train datasets
from sklearn.model_selection import train_test_split

X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# scale non one-hot-encoded (numeric) columns only
# fit on train, then apply same scaler to test to avoid data leakage
from sklearn.preprocessing import StandardScaler

non_encoded_cols = [col for col in X_train.columns if not col.startswith('ocean_proximity_')]

scaler = StandardScaler()
X_train.loc[:, non_encoded_cols] = scaler.fit_transform(X_train[non_encoded_cols])
X_test.loc[:, non_encoded_cols] = scaler.transform(X_test[non_encoded_cols])

X_train.head()

In [ ]:
# train linear regression model
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)
print("Coefficients:")
for col, coef in zip(X_train.columns, model.coef_):
    print(f"  {col}: {coef:.2f}")

In [ ]:
# evaluate model on test set
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2:.4f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")